In [1]:
import os
import avstack
import avapi
import numpy as np
import cv2  
from PIL import Image 
from IPython.display import Video


%load_ext autoreload
%autoreload 2

# Importing data
data_base = '../../data'
obj_data_dir_k = os.path.join(data_base, 'KITTI/object')
raw_data_dir_k = os.path.join(data_base, 'KITTI/raw') 
data_dir_n     = os.path.join(data_base, 'nuScenes')
data_dir_c     = os.path.join(data_base, 'CARLA/ego-lidar')

# Instantiating Scene Manager
KSM = avapi.kitti.KittiScenesManager(obj_data_dir_k, raw_data_dir_k, convert_raw=False)
NSM = avapi.nuscenes.nuScenesManager(data_dir_n)
CSM = avapi.carla.CarlaScenesManager(data_dir_c)



Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [39]:
def make_movie(DM, dataset_name, save_movie=False):
    model = '2d-img'
    if model == '2d-img':
        M = avstack.modules.perception.object2dfv.MMDetObjectDetector2D(
            model='fasterrcnn', dataset=dataset_name, gpu=1)
    elif model == '3d-img':
        M = avstack.modules.perception.object3d.MMDetObjectDetector3D(
            model='pgd', dataset=dataset_name, gpu=0)
    elif model == '3d-lidar':
        M = avstack.modules.perception.object3d.MMDetObjectDetector3D(
            model='pointpillars', dataset=dataset_name, gpu=1)
    else:
        raise NotImplementedError(model)
    
    if DM.frames is None:
        print("NO FRAMES in Scene")
        return 
        
    range_frames = range(len(DM.frames))
    CAM = 'image-2'
    if (dataset_name == 'carla'):
        range_frames = range(4, len(DM.frames)-5)
        CAM = 'CAM_FRONT'
    if (dataset_name == 'nuscenes'):
        CAM = 'main_camera'        
        
    imgs = []
    for frame_idx in range_frames:
        frame = DM.get_frames(sensor="main_camera")[frame_idx]
        objects = DM.get_objects(frame, sensor='main_lidar')
        img = DM.get_image(frame, sensor=CAM)
        if model == '2d-img':
            outputs = M(img, identifier='test', frame=frame)
        elif model == '3d-img':
            outputs = M(img, identifier='test', frame=frame)
        elif model == '3d-lidar':
            outputs = M(pc, identifier='test', frame=frame)
        else:
            raise NotImplementedError(model)
            
        imgs.append(avapi.visualize.snapshot.show_image_with_boxes(img, outputs.data, inline=False, return_images=True))

    # generate movie
    movie_name = dataset_name + '_scene_movie.mp4'
  
    height, width, layers = imgs[0].shape
    size = (width,height)
    video = cv2.VideoWriter(movie_name,cv2.VideoWriter_fourcc(*'DIVX'), 15, size)
    for img in imgs:
        video.write(img)  
    
    # Deallocating memories taken for window creation 
    cv2.destroyAllWindows()  
    print("Scene video:")
    
    video.release()  # releasing the video generated 
        
   
  
        





    
    

In [40]:
KDM = KSM.get_scene_dataset_by_index(scene_idx=0)
CDM = CSM.get_scene_dataset_by_index(scene_idx=0)
NDM = NSM.get_scene_dataset_by_index(scene_idx=0)
make_movie(NDM, 'nuscenes')

FileNotFoundError: Cannot find work_dirs/nuscenes/faster_rcnn_r50_fpn_1x_nuscenes.pth checkpoint